# Paper #49 — Global Seismology of the Sun: Implementation
## 전역 태양지진학 구현

**Reference**: Basu, S., *Living Reviews in Solar Physics* **13**, 2 (2016).

This notebook reproduces the key quantitative tools of global helioseismology discussed in Basu (2016):

1. **l-ν ridge diagram** — asymptotic p-mode dispersion $\omega \approx (n+1/2)\pi c_0/\int dr/c$
2. **Power spectrum** with Lorentzian mode profiles
3. **Duvall's Law** empirical relation
4. **Rotational splitting** pattern for a single multiplet
5. **Sound-speed reconstruction** from a Model-S-like profile
6. **Tachocline signature** in Ω(r) inversion

---

본 노트북은 Basu (2016)의 핵심 정량 도구들을 Python으로 재현한다:

1. **l-ν 능선 다이어그램** — p-mode 점근 분산관계
2. **전력 스펙트럼** (Lorentzian 선형 이용)
3. **Duvall 법칙** — 경험 관계
4. **회전 분열** 합성 패턴 (단일 다중항)
5. **표준 음속 프로파일** (Model S 근사) 재구성
6. **타코클린** 특징을 담은 Ω(r) 역전

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid, quad

# Reproducibility
np.random.seed(42)

# Solar constants (SI)
R_SUN = 6.957e8  # m
M_SUN = 1.989e30  # kg
G_GRAV = 6.674e-11  # m^3 kg^-1 s^-2

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## 1. Model-S-like Sound-Speed Profile / Model S 근사 음속 프로파일

We build a simple analytic approximation to the solar sound speed $c(r)$ that captures:
- Central value $\sim 500$ km/s
- Monotonic decrease through the radiative interior
- Sharp change at the convection-zone base $r_b = 0.713\,R_\odot$
- Steep drop in the outer convection zone
- Surface value $\sim 10$ km/s

This is used as a proxy for Model S in subsequent computations.

태양 음속의 해석적 근사를 구성한다: 중심 ~500 km/s, 대류대 바닥 r_b = 0.713 R_☉에서 급격 변화, 표면 ~10 km/s.

In [ ]:
def solar_sound_speed(r_frac: np.ndarray) -> np.ndarray:
    """Construct a Model-S-like sound speed profile.

    Args:
        r_frac: Fractional radius r/R_sun (array_like).

    Returns:
        Sound speed array in m/s with the same shape as r_frac.
    """
    r_frac = np.asarray(r_frac)
    c = np.zeros_like(r_frac, dtype=float)
    # Radiative zone: exponential fall from ~500 km/s at core to ~230 km/s at r_b
    rad = r_frac <= 0.713
    c[rad] = 500e3 * np.exp(-1.15 * r_frac[rad])
    # Convection zone: steeper drop to surface
    cz = (r_frac > 0.713) & (r_frac <= 0.999)
    c_b = 500e3 * np.exp(-1.15 * 0.713)
    c_surf = 1.0e4
    c[cz] = c_b * (c_surf / c_b) ** ((r_frac[cz] - 0.713) / (0.999 - 0.713))
    # Photosphere cap
    surf = r_frac > 0.999
    c[surf] = c_surf
    return c


r_frac = np.linspace(0.001, 0.999, 4000)
c_profile = solar_sound_speed(r_frac)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r_frac, c_profile / 1e3, 'k-', lw=1.5)
ax.axvline(0.713, color='red', ls='--', label=r'$r_b = 0.713\,R_\odot$')
ax.set_xlabel(r'$r / R_\odot$')
ax.set_ylabel(r'Sound speed $c$ (km/s)')
ax.set_title('Model-S-like Solar Sound Speed Profile / 태양 음속 프로파일')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Asymptotic p-mode Ridges in the l-ν Diagram / l-ν 다이어그램의 p-mode 능선

From Tassoul's asymptotic analysis (Eq. 68 of Basu 2016):

$$\nu_{n,\ell} \simeq \left(n + \frac{\ell}{2} + \alpha_p\right)\Delta\nu, \qquad \Delta\nu = \left(2\int_0^R \frac{dr}{c}\right)^{-1}.$$

We compute Δν for our c(r) and plot the ridges for n = 1..25 and ℓ = 0..150.

Tassoul 점근 공식으로 주파수 ν_{n,ℓ}의 능선을 그린다.

In [ ]:
def compute_large_separation(r_frac: np.ndarray, c: np.ndarray) -> float:
    """Compute the large frequency separation in microhertz.

    Uses Delta_nu = 1 / (2 * integral from 0 to R of dr / c).

    Args:
        r_frac: Fractional radius.
        c: Sound-speed profile in m/s.

    Returns:
        Large separation Delta_nu in microHz.
    """
    r_m = r_frac * R_SUN
    acoustic_time = np.trapz(1.0 / c, r_m)  # seconds
    delta_nu = 1.0 / (2.0 * acoustic_time)  # Hz
    return delta_nu * 1e6  # microHz


delta_nu = compute_large_separation(r_frac, c_profile)
print(f'Computed large separation: Delta_nu = {delta_nu:.1f} microHz')
print(f'Observed solar value: Delta_nu ~ 135 microHz')

# Ridges: nu_{n,l} = (n + l/2 + alpha_p) * Delta_nu
alpha_p = 1.5
l_values = np.arange(0, 151)
n_values = np.arange(1, 26)

fig, ax = plt.subplots(figsize=(7, 5))
for n in n_values:
    nu = (n + l_values / 2 + alpha_p) * delta_nu
    keep = (nu > 1000) & (nu < 5000)
    ax.plot(l_values[keep], nu[keep], 'b-', lw=0.6, alpha=0.7)

ax.set_xlim(0, 150)
ax.set_ylim(1000, 5000)
ax.set_xlabel(r'Spherical harmonic degree $\ell$')
ax.set_ylabel(r'Frequency $\nu$ ($\mu$Hz)')
ax.set_title(r'$\ell$-$\nu$ Ridge Diagram (Tassoul asymptotic) / p-mode 능선')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Power Spectrum with Lorentzian Mode Profiles / Lorentzian 선형의 전력 스펙트럼

Each observed p-mode appears in the oscillation power spectrum as a Lorentzian peak:

$$P(\nu) = \sum_{\text{modes}} \frac{A_i}{1 + [2(\nu - \nu_i)/\gamma_i]^2},$$

where $A_i$ is amplitude, $\gamma_i$ is FWHM linewidth, and a Gaussian envelope modulates amplitude around $\nu_{\max}\approx 3.1$ mHz.

관측된 각 p-mode는 Lorentzian 피크로 나타나며, ν_max 근처에 Gaussian 포락선 모양 진폭을 갖는다.

In [ ]:
def build_power_spectrum(
    mode_freqs: np.ndarray,
    amplitudes: np.ndarray,
    linewidths: np.ndarray,
    nu_grid: np.ndarray,
    noise_level: float = 0.05,
) -> np.ndarray:
    """Build a synthetic oscillation power spectrum of Lorentzian modes.

    Args:
        mode_freqs: Mode central frequencies in microHz.
        amplitudes: Mode peak powers (arbitrary units).
        linewidths: FWHM linewidths in microHz.
        nu_grid: Frequency grid for the power spectrum in microHz.
        noise_level: Relative background noise level.

    Returns:
        Power spectrum array evaluated on nu_grid.
    """
    power = np.zeros_like(nu_grid)
    for nu_i, a_i, g_i in zip(mode_freqs, amplitudes, linewidths):
        power += a_i / (1.0 + (2.0 * (nu_grid - nu_i) / g_i) ** 2)
    rng = np.random.default_rng(0)
    power += noise_level * rng.exponential(size=nu_grid.shape)  # chi-squared noise
    return power


# Take l=0 radial modes, n = 15..25 (centred around 3 mHz)
n_arr = np.arange(15, 26)
nu_modes = (n_arr + 0 + alpha_p) * delta_nu  # microHz
# Gaussian amplitude envelope centred at nu_max = 3100 microHz
nu_max = 3100.0
sigma_env = 500.0
amps = np.exp(-0.5 * ((nu_modes - nu_max) / sigma_env) ** 2)
# Line-widths grow with frequency (mode damping)
lwidths = 0.5 + 0.8 * (nu_modes / nu_max) ** 4  # microHz

nu_grid = np.linspace(1500, 4500, 20000)
power = build_power_spectrum(nu_modes, amps, lwidths, nu_grid)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(nu_grid, power, 'k-', lw=0.5)
ax.set_xlabel(r'Frequency $\nu$ ($\mu$Hz)')
ax.set_ylabel('Power (arb. units)')
ax.set_title('Synthetic p-mode Power Spectrum ($\\ell=0$) / 태양 p-mode 전력 스펙트럼')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Duvall's Law / Duvall 법칙

Duvall (1982) found that plotting $(n+\alpha_p)\pi/\omega$ vs. $w = \omega/L$ ($L = \ell + 1/2$) collapses all observed p-modes onto a single curve:

$$F(w) = \int_{r_t}^R \left(1 - \frac{L^2 c^2}{\omega^2 r^2}\right)^{1/2} \frac{dr}{c}.$$

We compute F(w) from our c(r) and over-plot the discrete mode points (n+α_p)π/ω.

Duvall 법칙: (n+α_p)π/ω vs w = ω/L에서 모든 p-mode가 하나의 곡선으로 모인다.

In [ ]:
def duvall_F(w: float, r_frac: np.ndarray, c: np.ndarray) -> float:
    """Compute Duvall's F(w) by numerical integration.

    F(w) = integral from r_t to R of sqrt(1 - c^2 / (w^2 r^2)) * dr / c.
    The lower turning point r_t is defined by c(r_t) / r_t = w.

    Args:
        w: Value of omega / L in m/s (angular frequency per inverse length).
        r_frac: Fractional radius grid.
        c: Sound speed profile in m/s.

    Returns:
        F(w) in seconds.
    """
    r_m = r_frac * R_SUN
    ratio = c / r_m  # c(r)/r
    # mode is propagating where c/r < w
    inside = ratio < w
    if not np.any(inside):
        return 0.0
    integrand = np.zeros_like(r_m)
    integrand[inside] = np.sqrt(1 - (ratio[inside] / w) ** 2) / c[inside]
    return np.trapz(integrand, r_m)


# Build a grid of w values
w_grid = np.linspace(50e3, 3000e3, 120)  # m/s
F_values = np.array([duvall_F(w, r_frac, c_profile) for w in w_grid])

# Produce discrete (n, l) mode points: nu_{n,l} = (n + l/2 + alpha_p) * Delta_nu
n_mesh, l_mesh = np.meshgrid(np.arange(5, 30), np.arange(1, 100), indexing='ij')
nu_mesh = (n_mesh + l_mesh / 2 + alpha_p) * delta_nu  # microHz
omega_mesh = 2 * np.pi * nu_mesh * 1e-6  # rad/s
L_mesh = l_mesh + 0.5
w_points = omega_mesh * R_SUN / L_mesh  # m/s (since w has units c/r*R ~ velocity)
F_points = (n_mesh + alpha_p) * np.pi / omega_mesh  # seconds

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(w_grid / 1e3, F_values, 'k-', lw=1.5, label=r'Continuous $F(w)$ from $c(r)$')
mask = (nu_mesh > 1000) & (nu_mesh < 5000)
ax.scatter(w_points[mask] / 1e3, F_points[mask], s=2, c='red', alpha=0.5,
           label='Discrete (n,$\\ell$) modes')
ax.set_xlabel(r'$w = \omega R_\odot / L$ (km/s)')
ax.set_ylabel(r'$(n+\alpha_p)\pi/\omega$ (s)')
ax.set_title("Duvall's Law — modes collapse onto single curve / Duvall 법칙")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Rotational Splitting of a Single Multiplet / 회전 분열

For a mode $(n, \ell)$ split by uniform rotation $\Omega_s$, each $m$-component shifts by

$$\delta\nu_m = m \frac{\Omega_s}{2\pi}.$$

For differential rotation, the shift is a kernel-weighted average of $\Omega(r,\theta)$ over the mode region. We simulate an $\ell=10$ multiplet for (a) uniform rotation at 460 nHz and (b) a latitude-dependent surface rotation.

회전에 의한 분열: 단일 다중항에서 m-성분마다 Δν = m·Ω_s/(2π) 만큼 이동한다.

In [ ]:
def splitting_pattern(l: int, omega_s_nhz: float, nu_center_uhz: float) -> np.ndarray:
    """Compute the (2l+1) frequency components of a rotationally split multiplet.

    Args:
        l: Spherical harmonic degree.
        omega_s_nhz: Angular rotation rate expressed as frequency in nHz.
        nu_center_uhz: Central (m=0) frequency in microHz.

    Returns:
        Array of 2*l+1 split frequencies in microHz.
    """
    m = np.arange(-l, l + 1)
    return nu_center_uhz + m * omega_s_nhz * 1e-3  # nHz -> microHz


l = 10
nu_c = 3100.0  # microHz
Omega_eq = 460.0  # nHz (solar equator)
freqs_uniform = splitting_pattern(l, Omega_eq, nu_c)

# Differential rotation -- simulate by using a_j expansion
# For latitude-dependent rotation, odd a-coefficients.
def differential_splitting(l: int, a1: float, a3: float,
                            nu_center_uhz: float) -> np.ndarray:
    """Compute splittings with odd a-coefficients a1 and a3.

    Uses polynomial expansion delta_nu_m = a1*P1(m) + a3*P3(m), with
    P_j approximated by Legendre-type polynomials in m/l.

    Args:
        l: Spherical harmonic degree.
        a1: First odd coefficient in nHz.
        a3: Third odd coefficient in nHz.
        nu_center_uhz: Central frequency in microHz.

    Returns:
        Array of 2*l+1 frequencies in microHz.
    """
    m = np.arange(-l, l + 1)
    x = m / l
    P1 = x
    P3 = 0.5 * (5 * x ** 3 - 3 * x)
    shift_nhz = a1 * P1 * l + a3 * P3 * l
    return nu_center_uhz + shift_nhz * 1e-3


freqs_diff = differential_splitting(l, a1=440.0, a3=-25.0, nu_center_uhz=nu_c)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
m_arr = np.arange(-l, l + 1)
ax1.vlines(freqs_uniform, 0, 1, color='blue')
ax1.set_title(f'Uniform rotation, $\\Omega_s/2\\pi$ = {Omega_eq} nHz\n균일 회전')
ax1.set_xlabel(r'$\nu$ ($\mu$Hz)')
ax1.set_ylabel('Amplitude (arb.)')
ax1.grid(True, alpha=0.3)

ax2.vlines(freqs_diff, 0, 1, color='red')
ax2.set_title('Differential rotation ($a_1$=440, $a_3$=−25 nHz)\n차등 회전')
ax2.set_xlabel(r'$\nu$ ($\mu$Hz)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Total splitting range (uniform): {(freqs_uniform[-1]-freqs_uniform[0])*1000:.1f} nHz')
print(f'Total splitting range (diff):    {(freqs_diff[-1]-freqs_diff[0])*1000:.1f} nHz')

## 6. Sound-Speed Reconstruction via Duvall Inversion (Abel) / Duvall 기반 음속 재구성

Gough (1984) showed Duvall's law yields an Abel-type integral (Eq. 85 of Basu 2016):

$$w^3 \frac{dF}{dw} = \int_w^{a_s}\left(1 - \frac{a^2}{w^2}\right)^{-1/2} \frac{d\ln r}{d\ln a}\, da.$$

Rather than solving the full inverse Abel equation, we demonstrate the **forward** sensitivity: given a parameterized c(r), compute F(w) and show how a 0.5% perturbation in the deep interior propagates. This mimics the structure inversion kernel sensitivity.

Gough 1984 결과: Duvall 법칙에서 Abel형 적분이 나온다. 여기서는 순방향 감도 (c(r) 섭동이 F(w)에 미치는 영향)로 음속 inversion의 민감도를 시연한다.

In [ ]:
def perturbed_sound_speed(r_frac: np.ndarray, amp: float = 0.005,
                           r_center: float = 0.5, r_width: float = 0.08) -> np.ndarray:
    """Return c(r) perturbed by a Gaussian bump.

    Args:
        r_frac: Fractional radius array.
        amp: Relative amplitude of the bump (e.g., 0.005 = 0.5%).
        r_center: Location of the Gaussian centre in r/R_sun.
        r_width: Gaussian sigma.

    Returns:
        Perturbed sound-speed array in m/s.
    """
    c0 = solar_sound_speed(r_frac)
    bump = amp * np.exp(-0.5 * ((r_frac - r_center) / r_width) ** 2)
    return c0 * (1.0 + bump)


c_perturbed = perturbed_sound_speed(r_frac, amp=0.005, r_center=0.5)
delta_nu_perturbed = compute_large_separation(r_frac, c_perturbed)

print(f'Baseline    Delta_nu = {delta_nu:.2f} microHz')
print(f'Perturbed   Delta_nu = {delta_nu_perturbed:.2f} microHz')
print(f'Relative change      = {(delta_nu_perturbed/delta_nu - 1)*100:+.3f} %')

F_perturbed = np.array([duvall_F(w, r_frac, c_perturbed) for w in w_grid])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(r_frac, c_profile / 1e3, 'k-', label='Baseline / 기준')
ax1.plot(r_frac, c_perturbed / 1e3, 'r--', label='Perturbed (+0.5%) / 섭동')
ax1.set_xlabel(r'$r/R_\odot$')
ax1.set_ylabel('c (km/s)')
ax1.set_title('Perturbed sound-speed / 섭동 음속')
ax1.legend()
ax1.grid(True, alpha=0.3)

diff_F = (F_perturbed - F_values) / F_values
ax2.plot(w_grid / 1e3, diff_F * 100, 'b-')
ax2.set_xlabel(r'$w$ (km/s)')
ax2.set_ylabel(r'$\delta F(w)/F(w)$ (%)')
ax2.set_title('Fractional change in F(w) / F 상대 변화')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Tachocline Signature in Rotation Profile / 타코클린 회전 프로파일

We construct a 2-D rotation rate $\Omega(r,\theta)$ that matches Fig. 26 of Basu (2016):

- Convection zone (r > 0.71): differential rotation with ~460 nHz at equator, ~330 nHz at pole.
- Radiative interior (r < 0.71): solid-body rotation at ~430 nHz.
- Thin shear layer **tachocline** at r = 0.71 R_☉ with thickness ~0.04 R_☉.

Ω(r,θ)를 구성하여 태양 회전 프로파일(Howe+ 2005)을 재현한다.

In [ ]:
def solar_rotation_profile(r_frac: np.ndarray, theta: np.ndarray,
                            r_tach: float = 0.713, width: float = 0.02,
                            Omega_rad: float = 430.0) -> np.ndarray:
    """Construct a 2-D solar rotation profile Omega(r, theta).

    Convection-zone component uses a surface-like law
    Omega_CZ(theta) = A + B*cos(theta)^2 + C*cos(theta)^4.
    Transition to solid-body rotation below r_tach uses a tanh profile
    to model the tachocline shear layer.

    Args:
        r_frac: 1-D fractional radius grid.
        theta: 1-D co-latitude array (radians).
        r_tach: Tachocline radius in r/R_sun.
        width: Half-width of the tachocline transition in r/R_sun.
        Omega_rad: Radiative-interior rotation rate in nHz.

    Returns:
        Array of shape (len(r_frac), len(theta)) with Omega in nHz.
    """
    A, B, C = 461.0, -61.0, -73.0  # nHz; canonical Snodgrass (1984)
    cos2 = np.cos(theta) ** 2
    Omega_cz = A + B * cos2 + C * cos2 ** 2
    R_grid, T_grid = np.meshgrid(r_frac, Omega_cz, indexing='ij')
    # Smooth tachocline transition
    weight_cz = 0.5 * (1 + np.tanh((r_frac[:, None] - r_tach) / width))
    return weight_cz * T_grid + (1 - weight_cz) * Omega_rad


r_grid = np.linspace(0.1, 0.98, 400)
theta_grid = np.linspace(0.01, np.pi / 2, 200)
Omega_2d = solar_rotation_profile(r_grid, theta_grid)

# Plot equator, 60 deg latitude, and pole cuts
theta_eq = 0
theta_60 = np.pi / 2 - np.radians(60)
theta_pole = 0.01

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
# Radial cuts
idx_eq = np.argmin(np.abs(theta_grid - (np.pi / 2)))  # equator at theta=pi/2
idx_60 = np.argmin(np.abs(theta_grid - (np.pi / 2 - np.radians(60))))
idx_po = np.argmin(np.abs(theta_grid - np.radians(10)))

ax1.plot(r_grid, Omega_2d[:, idx_eq], 'r-', lw=2, label='Equator / 적도')
ax1.plot(r_grid, Omega_2d[:, idx_60], 'g-', lw=2, label=r'$60^\circ$ / 60도')
ax1.plot(r_grid, Omega_2d[:, idx_po], 'b-', lw=2, label='Pole / 극')
ax1.axvline(0.713, color='k', ls='--', lw=0.8, alpha=0.6, label='Tachocline')
ax1.set_xlabel(r'$r/R_\odot$')
ax1.set_ylabel(r'$\Omega/2\pi$ (nHz)')
ax1.set_title('Rotation rate vs radius (3 latitudes) / 반지름 vs 회전')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2D map (quadrant)
R, T = np.meshgrid(r_grid, theta_grid, indexing='ij')
X = R * np.sin(T)
Y = R * np.cos(T)
pcm = ax2.pcolormesh(X, Y, Omega_2d, cmap='RdYlBu_r', shading='auto',
                      vmin=320, vmax=470)
circ_tach = plt.Circle((0, 0), 0.713, fill=False, color='k', ls='--', lw=1)
ax2.add_patch(circ_tach)
ax2.set_aspect('equal')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_xlabel(r'$r\sin\theta / R_\odot$')
ax2.set_ylabel(r'$r\cos\theta / R_\odot$')
ax2.set_title(r'$\Omega(r,\theta)/2\pi$ (nHz) / 2D 회전')
fig.colorbar(pcm, ax=ax2, label='nHz')

plt.tight_layout()
plt.show()

print(f'Equator surface rotation: {Omega_2d[-1, idx_eq]:.1f} nHz')
print(f'Pole surface rotation:    {Omega_2d[-1, idx_po]:.1f} nHz')
print(f'Radiative interior:       {Omega_2d[0, idx_eq]:.1f} nHz')

## 8. Summary / 요약

This notebook implemented:

1. A Model-S-like sound-speed profile c(r) with clean convection-zone boundary.
2. Computation of the **large separation** Δν ≈ 135 µHz directly from c(r).
3. The Tassoul p-mode **ridge diagram** in (ℓ, ν) space.
4. A synthetic **power spectrum** with Lorentzian mode peaks and Gaussian amplitude envelope around ν_max ≈ 3.1 mHz.
5. **Duvall's Law** evaluated for c(r) and the collapse of discrete (n,ℓ) modes onto it.
6. Sensitivity of F(w) to a 0.5% perturbation in deep c(r) — illustrating inversion kernels.
7. Rotationally split multiplets for uniform and differential rotation.
8. A 2-D Ω(r,θ) profile reproducing the **tachocline** at 0.71 R_☉ with equatorial ~460 nHz and polar ~330 nHz rotation above, and solid-body ~430 nHz below.

These are the building blocks of both helioseismic structure and rotation inversions discussed throughout Basu (2016).

이 노트북은 (1) Model S 근사 음속, (2) Δν 직접 계산, (3) l-ν 능선, (4) Lorentzian 선형의 전력 스펙트럼, (5) Duvall 법칙 순방향/역방향 감도, (6) 회전 분열 패턴, (7) 타코클린 포함 Ω(r,θ)를 구현하였다. 이들은 Basu (2016)의 구조·회전 역전의 기본 요소이다.